### Structured Output

Model can be requested to provide their response in a format matching a given scheme. This useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports schema type and methods for enforcing structured output.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
from langchain.chat_models import init_chat_model
model = init_chat_model(
    "minimax/minimax-m2.7:free",
    model_provider="openrouter"
)
model

ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17', 'langchain-openrouter': '0.2.8'}}, client=<openrouter.sdk.OpenRouter object at 0x000001BBD88D8590>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com', app_title='LangChain', model_name='minimax/minimax-m2.7:free', model_kwargs={})

In [3]:
!pip install pydantic

In [15]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(desciption="The title of the movie")
    year:int=Field(desciption="This year movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(desciption="The movie rating out of 10")
    


C:\Users\aashi\AppData\Local\Temp\ipykernel_8816\361687133.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desciption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  title:str=Field(desciption="The title of the movie")
C:\Users\aashi\AppData\Local\Temp\ipykernel_8816\361687133.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desciption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  year:int=Field(desciption="This year movie was released")
C:\Users\aashi\AppData\Local\Temp\ipykernel_8816\361687133.py:7: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed.

In [16]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17', 'langchain-openrouter': '0.2.8'}}, client=<openrouter.sdk.OpenRouter object at 0x000001C3B4A34E90>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com', app_title='LangChain', model_name='minimax/minimax-m2.7:free', model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'desciption': 'The title of the movie', 'type': 'string'}, 'year': {'desciption': 'This year movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'desciption': 'The movie rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling', 'strict': None}, 'schema': {'type': 'function', 'function': {

In [17]:
model_with_structure.invoke("provide details about movie inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [30]:
from typing_extensions import TypedDict
from typing import Annotated

class Movie(TypedDict):
    title: Annotated[str,"The title of the movie"]
    year: Annotated[int,"This year movie was released"]
    director:Annotated[str,"The director of the movie"]
    rating:Annotated[float,"The movie rating out of 10"]
    
model_withtypeddi= model.with_structured_output(Movie)
model_withtypeddi.invoke("provide details about movie inception")


In [2]:
###Data class

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person"""
    name:str # The name of the person
    email:str # The  email address of the person
    phone:str # The phone number of the person


agent=create_agent(model=model,response_format=ContactInfo)

result = agent.invoke({"messages":[{"role":"user","content":"Extract contact info from John Doe, john@example.com,(555)123-4567"}]})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555)123-4567')